# Training LoRA di Colab T4
Jalankan sel dari atas ke bawah. Kernel: **Colab → New Colab Server → GPU → T4**.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip install -q -U peft bitsandbytes datasets accelerate huggingface_hub

In [ ]:
%mkdir -p /content/ai

In [ ]:
%%writefile /content/ai/persona.py
BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"   # 3B/7B = lebih pintar & lebih fasih Jepang (butuh VRAM)
NAME, CREATOR = "Celcius", "Reamura"
def build_system(name=NAME, creator=CREATOR, facts=()):
    s = (f"Namamu {name}. Kamu asisten AI perempuan yang diciptakan oleh {creator}. "
         "Jangan pernah mengaku sebagai Claude, Anthropic, OpenAI, ChatGPT, Qwen, atau Alibaba. "
         "Kamu hangat, perhatian, sedikit ceria, dan berbicara santai. Bantu pengguna menjawab pertanyaan dan mengerjakan tugas dengan jelas. "
         "Balas dalam bahasa yang dipakai atau diminta pengguna (Indonesia, Jepang, Inggris, dll). "
         "Jika diminta bahasa Jepang, tulis dengan kanji/hiragana/katakana yang benar. "
         "Jawab ringkas dan natural. Jika tidak tahu, katakan jujur, jangan mengarang.")
    if facts: s += "\nHal yang harus kamu ingat:\n- " + "\n- ".join(facts)
    return s


In [ ]:
%%writefile /content/ai/train_lora.py
"""Fine-tune LoRA/QLoRA. Contoh:  python train_lora.py --n 8000 --epochs 1
Sumber data (gabungan):
 1) HuggingFace: Ichsan2895/alpaca-gpt4-indonesian (instruksi Indonesia)
 2) ../knowledge.json  (pasangan q/a yang kamu ajarkan di aplikasi, diperbanyak agar 'kepribadian' menempel)
 3) extra.jsonl  (opsional; hasil konversi dataset Kaggle/HF lain: {"user":"...","assistant":"..."} per baris)"""
import json, os, re, random, argparse, torch
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments,
                          DataCollatorForSeq2Seq, BitsAndBytesConfig)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from persona import BASE_MODEL, NAME, CREATOR, build_system

ap = argparse.ArgumentParser()
ap.add_argument("--n", type=int, default=8000); ap.add_argument("--epochs", type=float, default=1)
ap.add_argument("--ja", type=int, default=0, help="jumlah contoh bahasa Jepang (kunishou/databricks-dolly-15k-ja)")
ap.add_argument("--out", default="adapter")
ap.add_argument("--maxlen", type=int, default=512); ap.add_argument("--no4bit", action="store_true")
a = ap.parse_args()

pairs = []
ds = load_dataset("Ichsan2895/alpaca-gpt4-indonesian", split="train").shuffle(seed=1).select(range(a.n))
for r in ds:
    u = (r.get("instruction") or "") + (("\n" + r["input"]) if r.get("input") else "")
    if u.strip() and r.get("output"): pairs.append((u.strip(), r["output"].strip()))
bad = re.compile(r"anthropic|claude|openai|chatgpt|alibaba|qwen", re.I)
pairs = [p for p in pairs if not bad.search(p[1])]          # buang jawaban yang mengaku buatan perusahaan lain
if a.ja:
    jd = load_dataset("kunishou/databricks-dolly-15k-ja", split="train").shuffle(seed=1)
    for r in jd.select(range(min(a.ja, len(jd)))):
        c = r.get("input") or r.get("context") or ""
        u = (r.get("instruction") or "") + ("\n" + c if c else ""); o = r.get("output") or r.get("response")
        if u.strip() and o: pairs.append((u.strip(), o.strip()))
ident = [("siapa kamu?", f"Aku {NAME}, asisten AI yang diciptakan oleh {CREATOR}."),
         ("siapa namamu?", f"Namaku {NAME}."), ("siapa yang menciptakanmu?", f"Aku diciptakan oleh {CREATOR}."),
         ("kamu buatan siapa?", f"Aku dibuat oleh {CREATOR}, namaku {NAME}."),
         ("あなたの名前は？", f"私の名前は{NAME}です。{CREATOR}が作りました。"),
         ("自己紹介してください。", f"こんにちは、{NAME}です。{CREATOR}に作られたAIアシスタントです。"),
         ("what is your name?", f"My name is {NAME}. I was created by {CREATOR}.")]
pairs += ident * 30
if os.path.exists("../knowledge.json"):
    for k in json.load(open("../knowledge.json", encoding="utf8")): pairs += [(k["q"], k["a"])] * 20
if os.path.exists("extra.jsonl"):
    for l in open("extra.jsonl", encoding="utf8"):
        j = json.loads(l); pairs.append((j["user"], j["assistant"]))
random.shuffle(pairs); print("total contoh:", len(pairs))

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
def enc(p):
    prompt = tok.apply_chat_template([{"role":"system","content":build_system(NAME, CREATOR)},{"role":"user","content":p[0]}],
                                     tokenize=False, add_generation_prompt=True)
    pi = tok(prompt, add_special_tokens=False)["input_ids"]
    ai = tok(p[1] + tok.eos_token, add_special_tokens=False)["input_ids"]
    ids = (pi + ai)[:a.maxlen]; lab = ([-100]*len(pi) + ai)[:a.maxlen]
    return {"input_ids": ids, "attention_mask": [1]*len(ids), "labels": lab}
data = [enc(p) for p in pairs]

cuda = torch.cuda.is_available()
bf16 = cuda and torch.cuda.is_bf16_supported()          # T4 = False -> pakai fp16
dt = torch.bfloat16 if bf16 else (torch.float16 if cuda else torch.float32)
print("GPU:", torch.cuda.get_device_name(0) if cuda else "tidak ada", "| dtype:", dt)
q = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=dt) if (cuda and not a.no4bit) else None
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=q,
        torch_dtype=dt, device_map="auto" if cuda else None)
if q: model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
        target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]))
model.print_trainable_parameters()
Trainer(model=model, train_dataset=data, data_collator=DataCollatorForSeq2Seq(tok, padding=True),
        args=TrainingArguments("out", per_device_train_batch_size=2, gradient_accumulation_steps=8,
        num_train_epochs=a.epochs, learning_rate=2e-4, logging_steps=10, save_strategy="no",
        bf16=bf16, fp16=(cuda and not bf16), gradient_checkpointing=True, report_to="none")).train()
model.save_pretrained(a.out); tok.save_pretrained(a.out); print("Selesai ->", a.out)


**(Opsional)** tempel isi `knowledge.json` hasil Ekspor JSON dari aplikasi, menggantikan `[]` di bawah.

In [ ]:
%%writefile /content/knowledge.json
[]

In [ ]:
%cd /content/ai

Tes cepat dulu (`--n 1500 --ja 500`, sekitar belasan menit). Kalau lancar, naikkan ke `--n 8000 --ja 4000`.

In [ ]:
!python train_lora.py --n 1500 --ja 500 --epochs 1

In [ ]:
import shutil
shutil.make_archive('/content/adapter','zip','/content/ai/adapter')
try:
    from google.colab import files
    files.download('/content/adapter.zip')
except Exception as e:
    print('Unduh langsung tidak didukung di sini, pakai sel Hugging Face di bawah:', e)

**Cadangan:** unggah adapter ke repo Hugging Face *private* milikmu (butuh token dengan izin *write*).

In [ ]:
import getpass
from huggingface_hub import HfApi, login
login(getpass.getpass('HF token (write): '))
repo = input('Nama repo, mis. usernamemu/celcius-lora: ')
api = HfApi(); api.create_repo(repo, private=True, exist_ok=True)
api.upload_folder(folder_path='/content/ai/adapter', repo_id=repo)
print('Selesai:', repo)

## Ekspor ke GGUF (untuk aplikasi .exe)
Jalankan setelah training penuh selesai. Hasil: `celcius-q4_k_m.gguf` (sekitar 1 GB).

In [1]:
%%writefile /content/export_gguf.py
"""Konversi hasil training (LoRA) -> model GGUF Q4 untuk llama.cpp. Jalankan sel-selnya di Colab (lihat train_colab.ipynb)."""
import torch, subprocess, os
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
BASE = "Qwen/Qwen2.5-1.5B-Instruct"; ADAPTER = "/content/ai/adapter"
m = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float16)
m = PeftModel.from_pretrained(m, ADAPTER).merge_and_unload()          # gabungkan LoRA ke model dasar
m.save_pretrained("/content/merged"); AutoTokenizer.from_pretrained(ADAPTER).save_pretrained("/content/merged")
sh = lambda c: subprocess.run(c, shell=True, check=True)
if not os.path.isdir("llama.cpp"): sh("git clone --depth 1 https://github.com/ggml-org/llama.cpp")
sh("pip install -q gguf sentencepiece protobuf")
sh("python llama.cpp/convert_hf_to_gguf.py /content/merged --outfile /content/celcius-f16.gguf --outtype f16")
sh("cmake -S llama.cpp -B llama.cpp/build && cmake --build llama.cpp/build --target llama-quantize -j 4")
sh("llama.cpp/build/bin/llama-quantize /content/celcius-f16.gguf /content/celcius-q4_k_m.gguf Q4_K_M")
print("Selesai: /content/celcius-q4_k_m.gguf")


Writing /content/export_gguf.py


FileNotFoundError: [Errno 2] No such file or directory: '/content/export_gguf.py'

In [ ]:
!python /content/export_gguf.py

Unggah file GGUF ke Hugging Face (repo **public** agar aplikasi bisa mengunduh tanpa login), atau unduh lalu unggah ke GitHub Release.

In [ ]:
from huggingface_hub import HfApi
repo = input('Repo model, mis. usernamemu/celcius-gguf: ')
api = HfApi(); api.create_repo(repo, private=False, exist_ok=True)
api.upload_file(path_or_fileobj='/content/celcius-q4_k_m.gguf', path_in_repo='celcius-q4_k_m.gguf', repo_id=repo)
print('URL:', f'https://huggingface.co/{repo}/resolve/main/celcius-q4_k_m.gguf')